# Gobernanza Semántica y Knowledge Graphs para Discovery y Reutilización (Estratégico)

## Objetivos
- Establecer un marco de gobernanza para metadatos semánticos a escala.
- Definir arquitectura de referencia para discovery/reuse basada en KG.
- Medir impacto: métricas de descubrimiento, reutilización y tiempo a valor.
- Aplicar validaciones y políticas (SHACL/policy-as-graph).
- Documentar decisiones con ADRs.

## 1. Marco de Gobernanza Semántica
### Principios
- **Meaning-first**: el significado precede al esquema físico.
- **Domain ownership**: dominios gestionan conceptos bajo estándares comunes.
- **Interoperabilidad**: vocabularios compartidos (DCAT, SKOS, schema.org).
- **Automatización**: validaciones en CI/CD (semantic gates).
- **Observabilidad**: métricas de uso, descubrimiento y reuso.

### Roles
- Semantic Steward (por dominio)
- Ontology Lead (empresa)
- Catalog Owner / Platform Team
- Data Product Owners

### Procesos
- Alta/cambio/baja de conceptos y relaciones (RFC semántico)
- Aprobación de mapeos `mapsToConcept` y etiquetas PII/PHI
- Versionado y compatibilidad (deprecations)

## 2. Arquitectura de Referencia (Catalog + KG + Search API)
```
        Producers             Catalog             KG Store              Search/API
          |                    |                    |                      |
    Data Contracts  --->  Asset Registry  --->  Semantic Graph  --->  Discovery/Reuse
          |                    |                    |                      |
    pipelines CI/CD      metadata sync       SHACL/Policies        SDK/CLI/Portal
```
Tecnologías posibles: DataHub/Atlas (+ OpenLineage) + RDF store (Neptune/Blazegraph/Jena) + index (OpenSearch) + API Graph/REST.

## 3. Métricas de Discovery/Reutilización
- Search Success Rate (SSR)
- Average Time-to-Discover (ATD)
- Reuse Ratio (RR): consultas o pipelines que reutilizan datasets existentes
- Duplicate Concept Ratio (DCR)
- Concept Coverage (CC)

In [ ]:
# Simulador de métricas de discovery y reutilización
import random, statistics
random.seed(7)

def simulate_metrics(n_searches=200):
    success = [random.random() < 0.78 for _ in range(n_searches)]
    time_to_discover = [random.uniform(0.5, 3.5) for _ in range(n_searches)]  # horas
    reuse_ratio = 0.62  # % de consultas que reutilizan datasets existentes
    return {
        'SSR': sum(success)/len(success)*100,
        'ATD': statistics.mean(time_to_discover),
        'RR': reuse_ratio*100
    }

m = simulate_metrics()
print('=== Discovery & Reuse Metrics ===')
for k,v in m.items():
    print(f'{k}: {v:.1f}' + (' %' if k in ('SSR','RR') else ' h'))

def target_eval(metrics):
    targets = {'SSR': 85, 'ATD': 2.0, 'RR': 70}
    evals = []
    evals.append('SSR OK' if metrics['SSR']>=targets['SSR'] else 'SSR BAJO')
    evals.append('ATD OK' if metrics['ATD']<=targets['ATD'] else 'ATD ALTO')
    evals.append('RR OK' if metrics['RR']>=targets['RR'] else 'RR BAJO')
    return evals

print('Diagnóstico:')
for r in target_eval(m):
    print('-', r)

## 4. Políticas y Validación (SHACL)
Ejemplo de shape (conceptual) que exige que toda `Column` mapeada a `PII` tenga `maskingPolicy`.

```turtle
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix ex: <https://example.com/data/> .
ex:PIIMaskingShape a sh:NodeShape ;
  sh:targetClass ex:Column ;
  sh:property [
    sh:path ex:mapsToConcept ;
    sh:hasValue ex:concept/pii ;
  ] ;
  sh:property [
    sh:path ex:maskingPolicy ;
    sh:minCount 1 ;
  ] .
```

Esto puede ejecutarse con motores SHACL (p.ej., `pyshacl`) en CI/CD para bloquear despliegues que incumplen.

## 5. Caso de Estudio y ADR
**Contexto:** duplicidad de datasets cliente, baja reutilización, búsquedas ineficientes.
**Decisión:** implantar KG con mapeos `mapsToConcept`, facetas por dominio/concepto, SHACL para PII.
**Consecuencias:** (+) Reutilización ↑, tiempo de descubrimiento ↓; (–) inversión inicial, curva de aprendizaje.
**Métricas objetivo (12 meses):** SSR ≥ 90%, ATD ≤ 1.5h, RR ≥ 75%.
**ADR-020 (resumen):** Adoptar KG + SHACL + search semántico como capa de discovery corporativa.

## 6. Roadmap de Implementación
- Fase 1: mapeo de conceptos core (Customer, Product, Order) y 10 datasets clave.
- Fase 2: integración catálogo, generación automática de `mapsToConcept` desde contratos.
- Fase 3: SHACL en CI/CD, facetas semánticas en el portal, KPIs de discovery/reuse en dashboard.
- Fase 4: expansión a dominios restantes y automatización de recomendaciones por KG.